# Phase 3: Borrower Financial Analysis

Phase 3 consumes canonical fields extracted from the lender-style documents in `data/realistic_pdfs`. It routes the borrower to an income specialist, verifies assets, and extracts recurring credit obligations.

```text
Phase 2 state → borrower routing → income specialist → assets → liabilities
```

In [ ]:
# Locate the src-layout project and import the completed earlier phases.
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd() / "underwritingAgent", Path.cwd().parent / "underwritingAgent"]
PROJECT_ROOT = next(path.resolve() for path in candidates if (path / "pyproject.toml").exists())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

PDF_ROOT = PROJECT_ROOT / "data" / "realistic_pdfs"
GUIDELINES = PROJECT_ROOT / "data" / "underwriting_guidelines.jsonl"
INTAKE_REFERENCES = ["UW-26-0417-A", "BRK-90831", "WHL-77-2206"]
print(PROJECT_ROOT)


In [ ]:
from underwriting_agent.document_layer import build_document_workflow
from underwriting_agent.borrower_analysis import build_borrower_workflow
from underwriting_agent.intake_packages import resolve_document_paths

def through_phase_3(loan_id):
    paths = resolve_document_paths(PDF_ROOT, loan_id)
    state = build_document_workflow().invoke({
        "loan_id": loan_id,
        "document_paths": [str(path) for path in paths],
        "workflow_status": "INTAKE",
    })
    return build_borrower_workflow().invoke(state)


## 1. Inspect the realistic Phase 2 handoff

The wage-earner package contains a W-2, bank statement, merged credit report, long contract, and appraisal. Phase 2 normalizes the different layouts once; Phase 3 consumes those fields.

In [ ]:
paths = resolve_document_paths(PDF_ROOT, "UW-26-0417-A")
phase2_state = build_document_workflow().invoke({
    "loan_id": "UW-26-0417-A",
    "document_paths": [str(path) for path in paths],
    "workflow_status": "INTAKE",
})
for document in phase2_state["parsed_documents"]:
    print(document.document_type.value, document.extracted_fields)


## 2. Route the borrower

Employment type comes from application content. A conditional edge selects exactly one of the salaried, self-employed, mixed-income, or unsupported paths.

In [ ]:
from underwriting_agent.borrower_analysis import classify_borrower_node, route_borrower

route_update = classify_borrower_node(phase2_state)
print(route_update)
print("Next node:", route_borrower({**phase2_state, **route_update}))


## 3. Run the complete Phase 3 subgraph

In [ ]:
phase3 = build_borrower_workflow()
result = phase3.invoke(phase2_state)
print(result["income_analysis"].model_dump(mode="json"))
print(result["asset_analysis"].model_dump(mode="json"))
print(result["liability_analysis"].model_dump(mode="json"))


## 4. Compare the three realistic packages

In [ ]:
portfolio = []
for reference in INTAKE_REFERENCES:
    result = through_phase_3(reference)
    portfolio.append({
        "intake_reference": reference,
        "borrower_path": result["borrower_path"].value,
        "qualifying_income": result["income_analysis"].qualifying_monthly_income,
        "verified_assets": result["asset_analysis"].verified_assets,
        "monthly_debt": result["liability_analysis"].total_monthly_debt,
        "exceptions": result["income_analysis"].exceptions + result["asset_analysis"].exceptions,
    })
portfolio


## Phase 3 handoff

Phase 4 receives typed income, asset, and liability analyses with document provenance. Missing evidence remains visible for orchestration and human review.

## Optional production services: OpenAI and Pinecone

This phase intentionally remains deterministic. An OpenAI model may normalize messy source documents in Phase 2 and draft reviewer-facing prose in Phase 7, while borrower routing, income selection, asset exclusion, appraisal comparison, reconciliation, and exception rules remain typed Python logic. Pinecone is used only for guideline retrieval in Phase 5.